In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from transformers import EncodecModel
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
import torch.nn as nn
import utils as u


In [2]:
seed =1234
trace_len =1000
embedded_input = True # if True it embedds the input, if false it uses the original model
model_gcn=True
few_shot=False
u.seed_everything(seed)

Setting seeds


In [3]:
print(torch.cuda.is_available())
if torch.cuda.is_available():  
    dev = "cuda" 
    map_location=None
else:  
    dev = "cpu"  
    map_location='cpu'
device = torch.device(dev)

True


In [4]:
if few_shot:
    test_set_size = 0.89
else:
    test_set_size = 0.2
inputs = np.load('inputs_ci.npy', allow_pickle=True)
targets = np.load('targets.npy', allow_pickle=True)
metadata = np.load('meta.npy', allow_pickle=True)
graph_input = u.graph_creator(0.3)
graph_input = np.array([graph_input] * inputs.shape[0])
graph_features = np.load('station_coords.npy', allow_pickle=True)
graph_features = np.array([graph_features] * inputs.shape[0])


cutoff 0.3
(39, 3)
  Station  Latitude  Longitude
5    APEC  43.55846   12.41991
6    ATCC  43.18514   12.63994


In [5]:
if embedded_input:
    inputs_norm = u.normalize_for_emb(inputs[:, :, :trace_len, :])
    dset = TensorDataset(torch.tensor(inputs_norm, dtype=torch.float32), torch.tensor(targets, dtype=torch.float32),torch.tensor(graph_features, dtype=torch.float32),torch.tensor(graph_input, dtype=torch.float32), torch.tensor(metadata[:,:,0].astype(np.int64), dtype=torch.int))
    d_loader = DataLoader(dset, batch_size=1, shuffle=False)

    net_H =  EncodecModel.from_pretrained("facebook/encodec_24khz")
    net_E =  EncodecModel.from_pretrained("facebook/encodec_24khz")
    net_Z =  EncodecModel.from_pretrained("facebook/encodec_24khz")
    
    net_E.load_state_dict(torch.load("../models/STEAD_ch0.pth", map_location='cuda:0'))
    net_H.load_state_dict(torch.load("../models/STEAD_ch1.pth", map_location='cuda:0'))
    net_Z.load_state_dict(torch.load("../models/STEAD_ch2.pth", map_location='cuda:0'))
    
    net_H.to(device)
    net_E.to(device)
    net_Z.to(device)

    net_H.eval()
    net_E.eval()
    net_Z.eval()

    print('Encoding:')
    def hook(module, input, output):
        outputs["conv"] = output

    denc_X = []
    denc_target = []
    denc_features = []
    denc_graph = []
    denc_metadata = []

    hook_handle_E = net_E.encoder.layers[15].register_forward_hook(hook)
    hook_handle_H = net_H.encoder.layers[15].register_forward_hook(hook)
    hook_handle_Z = net_Z.encoder.layers[15].register_forward_hook(hook)

    with torch.no_grad():
        for idx, batch in enumerate(d_loader):
            reshaped_batch = batch[0].reshape(-1,trace_len,3).permute(0,2,1)
            E_btc = reshaped_batch[:,0,:].unsqueeze(1).float().to(device)
            H_btc = reshaped_batch[:,1,:].unsqueeze(1).float().to(device)
            Z_btc = reshaped_batch[:,2,:].unsqueeze(1).float().to(device)
            outputs = {}
            output = net_E(E_btc)
            out_E = outputs["conv"]

            outputs = {}
            output = net_H(H_btc)
            out_H = outputs["conv"]

            outputs = {}
            output = net_Z(Z_btc)
            out_Z = outputs["conv"]

            out_chs= torch.cat((out_E,out_H,out_Z), axis=1)

            denc_X.append(out_chs)
            denc_target.append(batch[1].reshape(-1,5))
            denc_features.append(batch[2].reshape(-1,2))
            denc_graph.append(batch[3].reshape(-1,39))
            denc_metadata.append(batch[4].reshape(-1,1))

    denc_X = torch.stack(denc_X)
    denc_target = torch.stack(denc_target)
    denc_features = torch.stack(denc_features)
    denc_graph = torch.stack(denc_graph)
    denc_metadata = torch.stack(denc_metadata)

    print("denc_X, target, feat, graph, metadata", denc_X.shape,denc_target.shape,denc_features.shape,denc_graph.shape, denc_metadata.shape)

The normalization is event-based, working on the 3 channels


c:\Users\Laura\Anaconda3\envs\spectrogram_zeus_gcn\lib\site-packages\torch\nn\utils\weight_norm.py:30: UserWarning: torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.
  warnings.warn("torch.nn.utils.weight_norm is deprecated in favor of torch.nn.utils.parametrizations.weight_norm.")


Encoding:
denc_X, target, feat, graph, metadata torch.Size([915, 39, 384, 4]) torch.Size([915, 39, 5]) torch.Size([915, 39, 2]) torch.Size([915, 39, 39]) torch.Size([915, 39, 1])


In [6]:
idx_dset=np.arange(len(inputs))
np.random.shuffle(idx_dset)
idx_dset=idx_dset.tolist()
idx_train_set=idx_dset[:len(idx_dset)-int(len(idx_dset)*test_set_size)]
idx_test_set=idx_dset[len(idx_dset)-int(len(idx_dset)*test_set_size):]
print("len(idx_dset)", len(idx_dset),"len(idx_train_set)", len(idx_train_set),"len(idx_test_set)", len(idx_test_set))

if embedded_input:
    enc_X = denc_X[idx_train_set]
    enc_target = denc_target[idx_train_set]
    enc_features = denc_features[idx_train_set]
    enc_graph = denc_graph[idx_train_set]
    enc_metadata = denc_metadata[idx_train_set]
    print("enc_X, target, feat, graph", enc_X.shape,enc_target.shape,enc_features.shape,enc_graph.shape)

    enc_test_X = denc_X[idx_test_set]
    enc_test_target = denc_target[idx_test_set]
    enc_test_features = denc_features[idx_test_set]
    enc_test_graph = denc_graph[idx_test_set]
    enc_test_metadata = denc_metadata[idx_test_set]
    print("enc_test_X, target, feat, graph", enc_test_X.shape,enc_test_target.shape,enc_test_features.shape,enc_test_graph.shape)
train_features = graph_features[idx_train_set]
test_features = graph_features[idx_test_set]
train_graph_input = graph_input[idx_train_set]
test_graph_input = graph_input[idx_test_set]
train_inputs = inputs[idx_train_set]
test_inputs = inputs[idx_test_set]
train_targets = targets[idx_train_set]
test_targets = targets[idx_test_set]
train_metadata = metadata[:,:,0].astype(np.int64)[idx_train_set]
test_metadata = metadata[:,:,0].astype(np.int64)[idx_test_set]

if embedded_input:
    train_inputs = u.normalize_for_emb(train_inputs[:, :, :trace_len, :])
    test_inputs = u.normalize_for_emb(test_inputs[:, :, :trace_len, :])
else:    
    train_inputs = u.normalize(train_inputs[:, :, :trace_len, :])
    test_inputs = u.normalize(test_inputs[:, :, :trace_len, :])

len(idx_dset) 915 len(idx_train_set) 732 len(idx_test_set) 183
enc_X, target, feat, graph torch.Size([732, 39, 384, 4]) torch.Size([732, 39, 5]) torch.Size([732, 39, 2]) torch.Size([732, 39, 39])
enc_test_X, target, feat, graph torch.Size([183, 39, 384, 4]) torch.Size([183, 39, 5]) torch.Size([183, 39, 2]) torch.Size([183, 39, 39])
The normalization is event-based, working on the 3 channels
The normalization is event-based, working on the 3 channels


In [7]:
# Create PyTorch datasets and dataloaders
train_dataset = TensorDataset(torch.tensor(train_inputs, dtype=torch.float32), torch.tensor(train_targets, dtype=torch.float32),torch.tensor(train_features, dtype=torch.float32),torch.tensor(train_graph_input, dtype=torch.float32), torch.tensor(train_metadata, dtype=torch.int))
test_dataset = TensorDataset(torch.tensor(test_inputs, dtype=torch.float32), torch.tensor(test_targets, dtype=torch.float32),torch.tensor(test_features, dtype=torch.float32), torch.tensor(test_graph_input, dtype=torch.float32), torch.tensor(test_metadata, dtype=torch.int))
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, drop_last=True)

In [10]:
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
batch_size = 20
num_epochs = 100
patience = 50  

best_losses_per_fold = [] 
if embedded_input:
    X_data=enc_X
    Y_data=enc_target
    feat_data=enc_features
    graph_input_data=enc_graph
    metadata_data=enc_metadata
else:
    X_data=torch.tensor(train_inputs, dtype=torch.float32)
    Y_data=torch.tensor(train_targets, dtype=torch.float32)
    feat_data=torch.tensor(train_features, dtype=torch.float32)
    graph_input_data=torch.tensor(train_graph_input, dtype=torch.float32)
    metadata_data=torch.tensor(train_metadata, dtype=torch.int)

for fold, (train_index, val_index) in enumerate(kf.split(X_data)):
    if embedded_input:
        if model_gcn:
            model= u.Model_gcn_for_embedding().to(device)
        else:
            model = u.Model_cnn_for_embedding().to(device)
    else:
        if model_gcn:
            model= u.OriginalModel_gcn().to(device)
        else:
            model= u.OriginalModel_cnn().to(device)
    best_model=type(model)().to(device)
    optimizer = torch.optim.RMSprop(model.parameters(), lr=0.0001, weight_decay=1e-4)  
    loss_function = nn.MSELoss()
    
    print(f"Starting fold {fold+1}/{n_splits}")
    
    best_val_loss = float('inf')
    best_val_losses = []
    epochs_without_improvement = 0 

    train_inputs, val_inputs = X_data[train_index], X_data[val_index]
    train_targets, val_targets  = Y_data[train_index], Y_data[val_index]
    train_features, val_features = feat_data[train_index], feat_data[val_index]
    train_graph_input, val_graph_input = graph_input_data[train_index], graph_input_data[val_index]
    train_metadata, val_metadata = metadata_data[train_index], metadata_data[val_index]

    train_dataset = TensorDataset(train_inputs, train_targets,train_features, train_graph_input, train_metadata)
    val_dataset = TensorDataset(val_inputs, val_targets, val_features, val_graph_input, val_metadata)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    for epoch in range(num_epochs):
        model.train()
        train_total_loss = 0 
        for inputs, targets, features, graph_input, metadatas in train_loader:
            optimizer.zero_grad()
            if embedded_input:
                inputs = inputs.permute(0,3,2,1) 
            pga_output, pgv_output, sa03_output, sa10_output, sa30_output = model(inputs.to(device), features.to(device), graph_input.to(device))
            loss = nn.MSELoss()(pga_output, targets[:, :, 0].to(device)) + nn.MSELoss()(pgv_output, targets[:, :, 1].to(device)) + nn.MSELoss()(sa03_output, targets[:, :, 2].to(device)) + nn.MSELoss()(sa10_output, targets[:, :, 3].to(device)) + nn.MSELoss()(sa30_output, targets[:, :, 4].to(device))
            loss.backward()
            optimizer.step()
            train_total_loss += loss.item()
        
        if epoch % 1 == 0:
            print(f"Train Epoch {epoch+1}, Loss: {train_total_loss/ len(train_loader)}")

        model.eval()
        val_total_loss = 0 
        with torch.no_grad():
            for inputs, targets, features, graph_input, metadatas in val_loader:
                if embedded_input:
                    inputs = inputs.permute(0,3,2,1) 
                pga_output, pgv_output, sa03_output, sa10_output, sa30_output = model(inputs.to(device), features.to(device), graph_input.to(device))
                loss = nn.MSELoss()(pga_output, targets[:, :, 0].to(device)) + nn.MSELoss()(pgv_output, targets[:, :, 1].to(device)) + nn.MSELoss()(sa03_output, targets[:, :, 2].to(device)) + nn.MSELoss()(sa10_output, targets[:, :, 3].to(device)) + nn.MSELoss()(sa30_output, targets[:, :, 4].to(device))
                val_total_loss += loss.item()
            
            avg_val_loss = val_total_loss/ len(val_loader)
            if epoch % 1 == 0:
                print(f"Val Epoch {epoch+1}, Loss: {avg_val_loss}")

            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                best_val_losses = [best_val_loss]
                best_model_path = "Checkpoint/model_embedding_"+str(embedded_input)+"_gcn_"+str(model_gcn)+"_few_"+str(few_shot)+"_fold_"+str(fold+1)+"_seed_"+str(seed)+".pth"
                torch.save(model.state_dict(), best_model_path)
                del best_model
                best_model = type(model)()
                best_model.load_state_dict(model.state_dict())
                print(f"New best model saved for fold {fold+1} with val loss: {best_val_loss}")
                epochs_without_improvement = 0 
            else:
                epochs_without_improvement += 1
            best_losses_per_fold.append(best_val_losses)
         
        model.eval()
        if embedded_input:
            test_dataset = TensorDataset(enc_test_X, enc_test_target,enc_test_features, enc_test_graph, enc_test_metadata)
        else:
            test_dataset = TensorDataset(torch.tensor(test_inputs, dtype=torch.float32), torch.tensor(test_targets, dtype=torch.float32), torch.tensor(test_features, dtype=torch.float32),  torch.tensor(test_graph_input, dtype=torch.float32), torch.tensor(test_metadata, dtype=torch.int))
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
        test_total_loss=0

        with torch.no_grad():
            for inputs, targets, features, graph_input, metadatas in test_loader:
                if embedded_input:
                    inputs = inputs.permute(0,3,2,1) 
                pga_output, pgv_output, sa03_output, sa10_output, sa30_output = model(inputs.to(device), features.to(device), graph_input.to(device))
                loss = nn.MSELoss()(pga_output, targets[:, :, 0].to(device)) + nn.MSELoss()(pgv_output, targets[:, :, 1].to(device)) + nn.MSELoss()(sa03_output, targets[:, :, 2].to(device)) + nn.MSELoss()(sa10_output, targets[:, :, 3].to(device)) + nn.MSELoss()(sa30_output, targets[:, :, 4].to(device))
                test_total_loss += loss.item()
            
            avg_test_loss = test_total_loss/ len(test_loader)
            if epoch % 1 == 0 or epochs_without_improvement == 0:
                print(f"Test Epoch {epoch+1}, Loss: {avg_test_loss}")
                
        if epochs_without_improvement >= patience:
                print(f"Early stopping triggered after {patience} epochs without improvement.")
                break

Starting fold 1/5
Train Epoch 1, Loss: 14.486053784688314
Val Epoch 1, Loss: 2.3207592368125916
New best model saved for fold 1 with val loss: 2.3207592368125916
Test Epoch 1, Loss: 2.599689853191376
Train Epoch 2, Loss: 1.8839631954828897
Val Epoch 2, Loss: 1.253733217716217
New best model saved for fold 1 with val loss: 1.253733217716217
Test Epoch 2, Loss: 1.4497975945472716
Train Epoch 3, Loss: 1.4055193781852722
Val Epoch 3, Loss: 1.1931503936648369
New best model saved for fold 1 with val loss: 1.1931503936648369
Test Epoch 3, Loss: 1.3212299466133117
Train Epoch 4, Loss: 1.2938597977161408
Val Epoch 4, Loss: 1.2200336456298828
Test Epoch 4, Loss: 1.3137146592140199
Train Epoch 5, Loss: 1.284704295794169
Val Epoch 5, Loss: 1.1106822341680527
New best model saved for fold 1 with val loss: 1.1106822341680527
Test Epoch 5, Loss: 1.2014465510845185
Train Epoch 6, Loss: 1.2396642804145812
Val Epoch 6, Loss: 1.1103799119591713
New best model saved for fold 1 with val loss: 1.1103799119

In [11]:
te_losses=[]
mse_lists=[]
rmse_lists=[]
mae_lists=[]
for fold in range(n_splits):
    del best_model
    best_model = type(model)()
    print("Model: Checkpoint/model_embedding_"+str(embedded_input)+"_gcn_"+str(model_gcn)+"_few_"+str(few_shot)+"_fold_"+str(fold+1)+"_seed_"+str(seed)+".pth")
    best_model.load_state_dict(torch.load("Checkpoint/model_embedding_"+str(embedded_input)+"_gcn_"+str(model_gcn)+"_few_"+str(few_shot)+"_fold_"+str(fold+1)+"_seed_"+str(seed)+".pth", map_location=torch.device(device)))
    best_model.to(device)               
    best_model.eval()

    if embedded_input:
        test_dataset = TensorDataset(enc_test_X, enc_test_target,enc_test_features, enc_test_graph, enc_test_metadata)
    else:
        test_dataset = TensorDataset(torch.tensor(test_inputs, dtype=torch.float32), torch.tensor(test_targets, dtype=torch.float32), torch.tensor(test_features, dtype=torch.float32),  torch.tensor(test_graph_input, dtype=torch.float32), torch.tensor(test_metadata, dtype=torch.int))
    
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    test_total_loss=0

    with torch.no_grad():
        predictions = []
        mse_list = []
        rmse_list = []
        mae_list = []

        for inputs, targets, features, graph_input, metadatas in tqdm(test_loader, total=len(test_loader)):
            if embedded_input:
                inputs = inputs.permute(0,3,2,1) 
            pga_output, pgv_output, sa03_output, sa10_output, sa30_output = best_model(inputs.to(device), features.to(device), graph_input.to(device))
            predictions.append(torch.stack([pga_output, pgv_output, sa03_output, sa10_output, sa30_output]))
            loss = nn.MSELoss()(pga_output, targets[:, :, 0].to(device)) + nn.MSELoss()(pgv_output, targets[:, :, 1].to(device)) + nn.MSELoss()(sa03_output, targets[:, :, 2].to(device)) + nn.MSELoss()(sa10_output, targets[:, :, 3].to(device)) + nn.MSELoss()(sa30_output, targets[:, :, 4].to(device))
            test_total_loss += loss.item()
        
        avg_te_loss = test_total_loss/ len(test_loader)
        print(f"Loss: {avg_te_loss}")
        predictions_dropLast = torch.stack(predictions[:-1])
        predictions_dropLast=predictions_dropLast.permute(0,2,3,1)
        predictions_dropLast=predictions_dropLast.reshape(-1,predictions_dropLast.shape[2],predictions_dropLast.shape[3]).cpu()

        for i in range(5):
            test_targets_dropLast=test_targets[:predictions_dropLast.shape[0]]
            
            mse_list.append(mean_squared_error(test_targets_dropLast[:, :, i], predictions_dropLast[:, :, i]))
            rmse_list.append(mean_squared_error(test_targets_dropLast[:, :, i], predictions_dropLast[:, :, i], squared=False))
            mae_list.append(mean_absolute_error(test_targets_dropLast[:, :, i], predictions_dropLast[:, :, i]))

        # Output results
        print('All averages:')
        print('MSE score:', np.mean(mse_list))
        print('RMSE score:', np.mean(rmse_list))
        print('MAE score:', np.mean(mae_list))
        mse_lists.append(mse_list)
        rmse_lists.append(rmse_list)
        mae_lists.append(mae_list)
        te_losses.append(avg_te_loss)
print("loss mean:", np.array(te_losses).mean())
print("mse mean: ", np.array(mse_lists).mean(),
      "PGA", f"{np.array([i[0] for i in mse_lists]).mean():.3f}",
      "PGV", f"{np.array([i[1] for i in mse_lists]).mean():.3f}",
      "PSA03", f"{np.array([i[2] for i in mse_lists]).mean():.3f}",
      "PSA1", f"{np.array([i[3] for i in mse_lists]).mean():.3f}",
      "PSA3", f"{np.array([i[4] for i in mse_lists]).mean():.3f}",)

print("rmse mean: ", np.array(rmse_lists).mean(),
      "PGA", f"{np.array([i[0] for i in rmse_lists]).mean():.3f}",
      "PGV", f"{np.array([i[1] for i in rmse_lists]).mean():.3f}",
      "PSA03", f"{np.array([i[2] for i in rmse_lists]).mean():.3f}",
      "PSA1", f"{np.array([i[3] for i in rmse_lists]).mean():.3f}",
      "PSA3", f"{np.array([i[4] for i in rmse_lists]).mean():.3f}",)

print("mae mean: ", np.array(mae_lists).mean(),
      "PGA", f"{np.array([i[0] for i in mae_lists]).mean():.3f}",
      "PGV", f"{np.array([i[1] for i in mae_lists]).mean():.3f}",
      "PSA03", f"{np.array([i[2] for i in mae_lists]).mean():.3f}",
      "PSA1", f"{np.array([i[3] for i in mae_lists]).mean():.3f}",
      "PSA3", f"{np.array([i[4] for i in mae_lists]).mean():.3f}",)

Model: Checkpoint/model_embedding_True_gcn_True_few_False_fold_1_seed_1234.pth


  0%|          | 0/10 [00:00<?, ?it/s]

Loss: 0.790483433008194
All averages:
MSE score: 0.14823756958059953
RMSE score: 0.37323448726709607
MAE score: 0.28047404810755533
Model: Checkpoint/model_embedding_True_gcn_True_few_False_fold_2_seed_1234.pth


  0%|          | 0/10 [00:00<?, ?it/s]

Loss: 0.8393224000930786
All averages:
MSE score: 0.1587139150827677
RMSE score: 0.39007188899570694
MAE score: 0.2932942210600158
Model: Checkpoint/model_embedding_True_gcn_True_few_False_fold_3_seed_1234.pth


...

Loss: 0.8355711758136749
All averages:
MSE score: 0.15115460571077627
RMSE score: 0.37798326344130795
MAE score: 0.2851033191242676
Model: Checkpoint/model_embedding_True_gcn_True_few_False_fold_4_seed_1234.pth


  0%|          | 0/10 [00:00<?, ?it/s]

Loss: 0.8430714786052704
All averages:
MSE score: 0.15571370039473326
RMSE score: 0.384481255036028
MAE score: 0.28927941254549994
Model: Checkpoint/model_embedding_True_gcn_True_few_False_fold_5_seed_1234.pth


  0%|          | 0/10 [00:00<?, ?it/s]

Loss: 0.8315573930740356
All averages:
MSE score: 0.15682228544023133
RMSE score: 0.38331876620489946
MAE score: 0.28912960913459196
loss mean: 0.8280011761188508
mse mean:  0.15412841524182164 PGA 0.161 PGV 0.140 PSA03 0.159 PSA1 0.146 PSA3 0.165
rmse mean:  0.38181793218900767 PGA 0.386 PGV 0.363 PSA03 0.390 PSA1 0.374 PSA3 0.396
mae mean:  0.28745612199438614 PGA 0.291 PGV 0.270 PSA03 0.298 PSA1 0.285 PSA3 0.294
